# Sesión 11 · Laura construye su bandeja multimedia verificable
**Universidad Central · Maestría en Analítica de Datos · Big Data 2026-2S**

## El encargo: ¿cómo recibe y prepara Compras Claras sus archivos de imagen y audio?

En las sesiones anteriores Laura organizó documentos, contrataciones y búsquedas. **En S10 únicamente alcanzamos los conceptos hasta la diapositiva 11**: imagen, señal, metadatos y la necesidad de conservar la procedencia. **No suponemos que ejecutaste ningún ETL ni que conoces Colab. Todo empieza aquí.**

El equipo de Compras Claras recibe **32 fotografías de un inventario editorial previamente adquirido** y **10 grabaciones de voz autorizadas de MINDS-14**. No están relacionadas entre sí y no acreditan hechos de contratación. Laura necesita un **servicio técnico común** que registre cada archivo sin inventar asociaciones.

**Encargo de negocio:** «Construye una bandeja donde una persona pueda identificar la fuente, abrir el original, recuperar una versión procesada, consultar sus características y comprobar que nadie alteró el contenido. Cuando crezca la carga, determina si más trabajadores ayudan o empeoran la velocidad».

### ¿Para qué lo hacemos?
- Para **no perder originales ni atribuciones**: cada objeto se guarda y se identifica.
- Para **hacer los medios utilizables**: obtener PNG normalizados, WAV PCM16 y métricas verificables.
- Para **encontrarlos después**: un catálogo consultable por archivo, fuente, modalidad y perfil.
- Para **entregar una operación fiable**: repetir, detectar corrupción y registrar duración/errores.
- Para **decidir cómo escalar**: medir lotes con 1, 2 y 4 threads. Un thread no equivale a un nodo distribuido.

**Producto final de Laura:** `raw/`, `derived/`, `outputs/catalogo.jsonl`, `outputs/catalogo.parquet`, `manifest_s11.json`, prueba de relectura, eventos de ejecución e informe de velocidad. El ZIP de clase conserva únicamente lo permitido por las licencias.

**Recursos:** [Presentación S11 y E2E local](https://jazaineam1.github.io/BigData2026/Presentaciones/s11-etl-audio-imagenes.html#s1) · **este cuaderno de código visible**. CPU gratuita; sin credenciales ni datos que debas traer. **No hay video** por adaptación docente al PDA; no sustituye el Taller 2 evaluativo.


## Cinco capítulos de una sola historia — una ruta que puedes ejecutar por tu cuenta

| Capítulo | ¿Qué necesita Laura? | Qué ejecutarás y comprobarás |
|---|---|---|
| **1. Recibir** | Dos tipos de fuentes con procedencia | Preparar Python → extraer RAW → leer nombre, origen y SHA |
| **2. Preparar** | Imágenes comparables y audios procesables, también largos | Transformar → observar píxeles, histogramas, forma de onda y RMS |
| **3. Encontrar** | Una bandeja que se pueda abrir días después | Cargar JSONL y Parquet → consultas → localizar archivos mediante rutas |
| **4. Escalar** | Saber qué ocurre con decenas y cientos de tareas | ETL en vivo → lotes → threads → medir velocidad y memoria |
| **5. Entregar** | Poder justificar que nada se perdió | Reintento → comparar hashes → manifiesto → exportar expediente |

**Recorrido mínimo:** ejecuta las celdas 0–7, realiza la **práctica de formatos** inmediatamente después de cargar el catálogo, ejecuta 8D–8G con `base` y termina 9A–9B. Los niveles de 102/332/1432 fuentes (8B–8C) son **profundización opcional**, no un requisito que bloquee la clase.

**Regla de autonomía:** cada celda te dirá **por qué**, **qué código ejecutas**, **qué salida exacta esperar**, **qué modificar**, **qué interpretar** y **qué hacer ante un error**. El código siempre está visible.

**Límite de la historia:** el `article_id` de una fotografía editorial no prueba una relación con el `source_record_id` de una voz de MINDS-14. La bandeja unifica el *contrato técnico*, no la semántica ni la propiedad intelectual.


## Antes del primer ▶: las palabras que usaremos para resolver el encargo

**Microejemplo:** Laura recibe `foto.webp`. Primero copia esos bytes a `raw/`, calcula SHA-256, genera `derived/foto_gris.png` y agrega una fila a `outputs/catalogo.jsonl`. Al final puede abrir la fotografía original y su derivado sin depender del cuaderno.

| Concepto | Qué es, para qué sirve y qué NO significa |
|---|---|
| **ETL** | **E**xtraer originales, **T**ransformarlos sin destruirlos, **L**oad/Cargar metadatos y derivados consultables. No equivale a clasificar automáticamente el significado. |
| **Archivo / bytes** | Un archivo son bytes almacenados. Su extensión (`.wav`, `.webp`) no siempre determina el códec real. |
| **RAW** | Bytes originales **inmutables**. Si un hash cambia, el proceso lo rechaza antes de sobrescribir. |
| **Derivado** | Copia transformada con parámetros explícitos: por ejemplo un `PNG` reducido o `WAV` PCM16 mono. Es distinta del RAW. |
| **Perfil** | «Receta» de la transformación: tamaño, escala de grises, tasa de audio o ganancia. El mismo RAW puede producir varios perfiles. |
| **Metadato** | Dato *sobre* un archivo: origen, nombre, formato, ruta, tamaño, duración o fecha. No es su contenido semántico. |
| **SHA-256 / hash** | Resumen de 64 dígitos hexadecimales que permite comparar **integridad de bytes**. No certifica que la noticia sea verdadera. |
| **JSON** | Texto con objetos, listas y pares `clave: valor`. Es útil para una ficha completa o un manifiesto jerárquico. |
| **JSONL** | *JSON Lines*: **un objeto JSON válido por línea**. Para agregar/leer registros uno a uno sin cargar un gran arreglo; cada salto de línea separa una fila. |
| **Parquet** | Formato **binario y columnar**: organiza por columnas, registra tipos y puede comprimir. Facilita consultar solo campos necesarios en grandes tablas. **No se abre como texto plano.** |
| **Catálogo** | Tabla donde cada fila identifica un activo/profil, su ruta, origen y métricas. La tabla no contiene necesariamente los bytes de la foto/audio. |
| **Manifiesto** | Inventario del paquete con rutas, conteos y hashes esperados. Lo usamos para verificar archivos al recuperar la entrega. |
| **Histograma** | Cuenta píxeles por rango de intensidad; no revela por sí solo los objetos de una imagen. |
| **RMS** | Raíz de la media de cuadrados de muestras acústicas; mide amplitud efectiva, no palabras ni intención. |
| **Frecuencia de muestreo** | Muestras por segundo, por ejemplo 16 000 Hz; no es un tono de 16 000 Hz. |
| **PCM16 / WAV** | PCM16 codifica amplitudes con 16 bits; WAV es un contenedor que puede guardar diferentes codificaciones. |
| **Bloque / chunk** | Fracción consecutiva de un audio o lote de fuentes. Permite trabajar con memoria acotada sin confundir **previsualizar** con **procesar todo**. |
| **Thread / hilo** | Trabajador concurrente dentro del mismo proceso Python. Comparte memoria y puede competir por CPU; no es un nodo Big Data. |
| **Partición / Parquet** | Subtabla guardada por cada lote; permite reabrir y verificar partes sin acumular todas en RAM. |
| **Throughput** | Rendimiento observado, por ejemplo «derivados por segundo». Se mide, no se adivina. |
| **Speedup** | Tiempo con un trabajador ÷ tiempo con varios. Si es menor que 1, la ejecución concurrente fue más lenta. |
| **Idempotencia** | Repetir con mismos originales y parámetros produce los mismos resultados, sin duplicaciones. |

**Ejemplos manuales:** luminancia media de cuatro píxeles `[0,64,128,255]` = 111,75. RMS de muestras `[0,1,0,-1]` = √(2/4) ≈ 0,707.

**Cómo se lee:** un registro del catálogo identifica un **derivado** de un original. **Qué nos dice:** el archivo tiene origen, formato y transformación trazables. **Qué NO permite concluir todavía:** qué objetos aparecen o qué palabras se dijeron. **Qué error común:** inventar `article_id` para las voces bancarias.


In [ ]:
#@title 0A · Preparar dependencias gratuitas (Colab o VS Code)
import importlib.util, subprocess, sys
paquetes = {"PIL":"pillow>=10,<13", "numpy":"numpy>=1.26,<3",
            "pandas":"pandas>=2.2,<3", "pyarrow":"pyarrow>=17,<24",
            "soundfile":"soundfile>=0.12,<1", "scipy":"scipy>=1.13,<2",
            "matplotlib":"matplotlib>=3.8,<4", "IPython":"ipython>=8", "ipywidgets":"ipywidgets>=8,<9", "sklearn":"scikit-learn>=1.5,<2"}
faltantes = [spec for modulo,spec in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltantes: subprocess.check_call([sys.executable,"-m","pip","install","-q",*faltantes])
import json, wave, hashlib, tempfile
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Audio
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
RAIZ = Path("/content/s11_etl_audio_imagenes") if EN_COLAB else Path.cwd()/"s11_etl_audio_imagenes"
RAIZ.mkdir(parents=True,exist_ok=True)
print("ENTORNO LISTO:", "Colab" if EN_COLAB else "VS Code/Jupyter", "·", RAIZ)


## 0C · ¿Qué contiene cada carpeta y qué necesita Laura recuperar?

La carpeta `s11_etl_audio_imagenes/` es **el espacio de trabajo temporal de Colab**. No basta con guardar el cuaderno: hay que descargar o conservar el expediente al acabar.

```text
s11_etl_audio_imagenes/
├── raw/
│   ├── imagenes/          Fotografías fuente WEBP, sin sobrescribir
│   └── audios/            WAV originales, aunque su códec no sea PCM16
├── derived/
│   ├── imagenes/          PNG transformados con un perfil registrado
│   └── audios/            WAV PCM16 mono 16 kHz procesados
├── cache/                 Parquet descargado de MINDS-14, verificado; evita repetir descarga
├── outputs/
│   ├── catalogo.jsonl     Una fila JSON por derivado
│   ├── catalogo.parquet   La misma tabla, codificada por columnas
│   ├── decision.json      Parámetros y alternativas que elegiste
│   ├── resumen.json       Conteos y datos de ejecución
│   ├── escala/            Particiones y métricas de lotes (opcional)
│   └── etl_live/          Eventos y comparaciones concurrentes (opcional)
├── manifest_s11.json      Inventario esperado y SHA-256 de originales/derivados
└── expediente_*.zip       Archivo portable para recuperar los resultados
```

**¿Por qué no guardamos todo junto?** Si `raw/` y `derived/` se mezclan, puedes sobrescribir un original y perder la evidencia. Si el catálogo está separado en `outputs/`, puedes consultarlo sin abrir cientos de archivos multimedia. La `cache/` permite evitar descargas repetidas, pero **no equivale al expediente final**.

**HAZ ESTO AHORA:** después de ejecutar «0A · Preparar» inspecciona las carpetas con la próxima celda. Algunas aún no existirán: se crean cuando ejecutas E, T o L. Comprueba al final que ya existen sus archivos.

**Cómo se lee:** cada ruta tiene una responsabilidad. **Qué nos dice:** dónde encontrar y recuperar un producto. **Qué NO permite concluir todavía:** que los archivos existan o sean válidos, hasta ejecutar el ETL y verificar sus hashes. **Qué error común:** entregar solo `.ipynb` y olvidar los bytes generados.


In [ ]:
# 0C · INSPECCIÓN REAL: ¿qué existe ya y qué se creará durante el ETL?
# Path representa una ruta. .exists() comprueba su existencia, no la validez del contenido.
for nombre in ["raw/imagenes", "raw/audios", "derived/imagenes",
               "derived/audios", "cache", "outputs"]:
    ruta = RAIZ / nombre
    print(f"{nombre:22s} → {'EXISTE' if ruta.exists() else 'PENDIENTE'}")

# Prueba de autonomía: reejecuta esta celda después de la etapa L y compara.


In [ ]:
#@title 0B · Cargar el motor abierto del repositorio (sin claves)
from urllib.request import urlopen, Request
rutas_locales = [Path("utils/s11_etl_core.py"), Path("../utils/s11_etl_core.py")]
archivo_motor = next((p for p in rutas_locales if p.is_file()), None)
if archivo_motor is None:
    archivo_motor = RAIZ/"s11_etl_core.py"
    origen = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/utils/s11_etl_core.py"
    if not archivo_motor.exists():
        with urlopen(Request(origen,headers={"User-Agent":"BigData-UCentral-S11/1.0"}),timeout=35) as r:
            archivo_motor.write_bytes(r.read())
import importlib.util as util
especificacion = util.spec_from_file_location("s11_etl_core",archivo_motor)
core = util.module_from_spec(especificacion)
especificacion.loader.exec_module(core)
assert core.ESQUEMA == "s11-etl-imagen-audio/v1"
print("MOTOR LISTO:",archivo_motor.name,"· contrato:",core.ESQUEMA)


## 1 · Tu decisión experimental: conserva una alternativa descartada

**HAZ ESTO AHORA.** Cambia como mínimo dos parámetros entre los perfiles disponibles y elige una alternativa y un límite de interpretación. No se pide redactar respuestas abiertas. Las opciones se registran en `outputs/decision.json`. Si una condición es inválida, el código la rechaza antes de descargar archivos.

**Cómo se lee:** los parámetros son parte del expediente. **Qué nos dice:** qué proceso produjo tu salida. **Qué NO permite concluir:** que el perfil elegido es siempre el óptimo. **Error común:** cambiar varios parámetros y atribuir a uno solo la diferencia observada.


In [ ]:
#@title 1 · Elige tu perfil y registra decisiones (modifica los valores)
PERFIL_IMAGEN = "grises" #@param ["grises","contraste","bordes"]
LADO_IMAGEN = 320 #@param [160,320,640] {type:"raw"}
GANANCIA_AUDIO = 1.0 #@param [0.5,1.0,1.5] {type:"raw"}
POSICION_VOZ = 1 #@param {type:"integer"}
ALTERNATIVA_DESCARTADA = 0 #@param [0,1,2,3] {type:"raw"}
LIMITE_ELEGIDO = 0 #@param [0,1,2,3] {type:"raw"}
# Alternativa: 1 = imagen sin reducción, 2 = audio sin conservar RAW, 3 = guardar solo CSV.
# Límite: 1 = RMS no transcribe, 2 = histograma no reconoce objetos, 3 = coincidencia de metadatos no prueba causalidad.
assert PERFIL_IMAGEN in ("grises","contraste","bordes")
assert LADO_IMAGEN in (160,320,640) and GANANCIA_AUDIO in (0.5,1.0,1.5)
assert type(POSICION_VOZ) is int and 1 <= POSICION_VOZ <= 10
assert ALTERNATIVA_DESCARTADA in (0,1,2,3) and LIMITE_ELEGIDO in (0,1,2,3)
DECISIONES = {"perfil_imagen":PERFIL_IMAGEN,"lado_imagen":LADO_IMAGEN,
 "ganancia_audio":GANANCIA_AUDIO,"voz_elegida":POSICION_VOZ,
 "alternativa_descartada":ALTERNATIVA_DESCARTADA,"limite_elegido":LIMITE_ELEGIDO}
print("DECISIONES ESTRUCTURADAS:",DECISIONES)


## 2 · E: 32 imágenes, misma historia editorial

**Problema de Laura:** los metadatos de las noticias y las fotografías tienen procedencia real. No es suficiente descargar una imagen y llamarla `foto.png`: necesitamos guardar bytes, identificador, crédito y SHA-256.

**HAZ ESTO AHORA.** Ejecuta la celda siguiente. Lee el inventario del snapshot docente (32 fotos, 24 artículos), descarga desde el **repositorio del curso**, verifica los 32 hashes antes de transformar y muestra tres créditos. No realiza solicitudes a los servidores del periódico.

**Salida:** `E · IMÁGENES: 32 RAW verificados`. Si no coincide la huella, el proceso se detiene; nunca reemplaza el original alterado.

**Cómo se lee:** una fila es una aparición de imagen en una noticia. **Qué nos dice:** hay 32 bytes originales distintos y 24 identificadores de artículo. **Qué NO permite concluir:** que las fotografías sean representativas del periódico o que prueben una relación con contratos. **Error común:** perder el crédito editorial al generar un thumbnail.


In [ ]:
#@title 2 · E: descargar y verificar el snapshot de 32 imágenes
IMAGENES_RAW = core.extraer_imagenes(RAIZ,max_imagenes=32)
assert len(IMAGENES_RAW)==32 and len({x["article_id"] for x in IMAGENES_RAW})==24
display(pd.DataFrame(IMAGENES_RAW)[["article_id","credit","original_sha256"]].head(5))
print("RAW INMUTABLE:",len(IMAGENES_RAW),"imágenes; artículos únicos:",24)


## 3 · E: diez voces humanas, una fuente diferente

**Problema de Laura:** medir la señal de voz sin confundirla con las noticias. MINDS-14 (PolyAI, CC BY 4.0) aporta diez grabaciones en español de distintas intenciones financieras. Cada WAV original se conserva byte por byte; su contenedor podría tener códec μ-law.

**HAZ ESTO AHORA.** Descarga una vez el Parquet auditado (~39 MB) desde Hugging Face, verifica su SHA-256 y selecciona de forma determinista una voz de cada una de diez clases distintas. La descarga puede tardar según tu conexión.

**Salida:** `E · AUDIOS: 10 voces reales` y una tabla con SHA, codec e identificador. Si la red falla, reejecuta; no reemplaces estas voces por tonos de prueba. No hay API paga ni credenciales.

**Cómo se lee:** cada `source_record_id` pertenece al corpus de voces, no a una noticia. **Qué nos dice:** el origen puede verificarse. **Qué NO permite concluir:** que todas las voces españolas o todas las intenciones están representadas por diez ejemplos. **Error común:** tratar el codec fuente como PCM16 sin decodificar.


In [ ]:
#@title 3 · E: diez voces reales, verificadas y sin duplicados
AUDIOS_RAW = core.extraer_audios(RAIZ,max_audios=10)
assert len(AUDIOS_RAW)==10 and len({x["original_sha256"] for x in AUDIOS_RAW})==10
assert all(x["article_id"] is None and x["license"]=="CC BY 4.0" for x in AUDIOS_RAW)
display(pd.DataFrame(AUDIOS_RAW)[["source_record_id","source_codec","source_rate_hz","license"]])
print("FUENTES SEPARADAS: imágenes con article_id; voces con source_record_id")


## 4A · Lee el código que realmente modifica píxeles

**¿Por qué?** Laura no puede afirmar «normalicé una imagen» sin poder mostrar qué se hizo a cada píxel. Vamos a construir un derivado simple desde **una fotografía real** antes de procesar el lote completo.

- `PIL.Image.open`: abre el archivo en modo lectura.
- `convert("L")`: convierte a una matriz de luminancia de un canal.
- `thumbnail((160,160))`: reduce proporcionalmente sin aumentar el original.
- `np.histogram`: cuenta píxeles por intervalos, no objetos reconocidos.
- `save`: guarda una **copia nueva**. RAW no se sobrescribe.

**HAZ ESTO AHORA:** ejecuta 2 · Extraer imágenes, después esta celda. Cambia el tamaño 160 a 320 y compara el histograma y la resolución. **Esperas ver:** `CÓDIGO DE IMAGEN REPRODUCIDO` y el archivo `outputs/tutorial_imagen.png`.

**Cómo se lee:** cada barra cuenta píxeles por luminancia. **Qué nos dice:** el efecto de la transformación. **Qué NO permite concluir:** objetos presentes, contratos o noticias. **Qué error común:** editar `raw/imagenes` en lugar de generar un derivado.


In [ ]:
# 4A · CÓDIGO VISIBLE: transformación real de UNA imagen verificada
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

archivo_raw = RAIZ / IMAGENES_RAW[0]["original_path"]  # archivo real, no imagen simulada
archivo_demo = RAIZ / "outputs" / "tutorial_imagen.png"
archivo_demo.parent.mkdir(parents=True, exist_ok=True)
with Image.open(archivo_raw) as original:
    imagen = original.convert("L")      # L: un canal de luminancia
    imagen.thumbnail((160, 160))        # no aumenta imágenes pequeñas
    imagen.save(archivo_demo)           # guarda DERIVADO, NO raw/

pixels = np.asarray(imagen)
histograma, limites = np.histogram(pixels, bins=8, range=(0,256))
assert sum(histograma) == pixels.size
assert core.sha256(archivo_raw) == IMAGENES_RAW[0]["original_sha256"]
print("CÓDIGO DE IMAGEN REPRODUCIDO ·", imagen.size, "·", pixels.size, "píxeles")
print("DERIVADO:", archivo_demo.relative_to(RAIZ))
fig,ax=plt.subplots(1,2,figsize=(10,3))
ax[0].imshow(imagen,cmap="gray",vmin=0,vmax=255)
ax[0].set_title("Derivado (no es el RAW)")
ax[1].bar(range(8),histograma)
ax[1].set_title("Histograma de 8 intervalos")
plt.tight_layout();plt.show()


## 4 · T: perfiles reales de imagen y conservación de píxeles

**HAZ ESTO AHORA.** El perfil elegido transforma las 32 imágenes con Pillow y conserva originales WEBP; todos los derivados son PNG. Cambia el perfil y/o el lado, reejecuta y compara el primer original con su derivado e histograma de ocho intervalos.

**Mini ficha:** `transformar_imagenes` recibe `raiz`, la lista de originales, `lado` (160/320/640) y `perfil` (grises/contraste/bordes); devuelve nuevos archivos y métricas. Error frecuente: pensar que bajar tamaño equivale a identificar objetos.

**Cómo se lee:** la suma de ocho contadores del histograma debe igualar `ancho × alto` del derivado. **Qué nos dice:** cantidad de píxeles e intensidad media. **Qué NO permite concluir:** qué institución u objeto aparece en la foto. **Error común:** comparar histogramas de imágenes con dimensiones distintas como si tuvieran el mismo número de muestras.
**Tu primer código:** ejecuta la práctica 4A con una imagen real, antes de transformar el lote con la función del motor. Abre el PNG y verifica que el RAW no cambió.


In [ ]:
#@title 4 · T: generar 32 imágenes derivadas e inspeccionar una
IMAGENES_T = core.transformar_imagenes(RAIZ, IMAGENES_RAW, lado=LADO_IMAGEN, perfil=PERFIL_IMAGEN)
assert len(IMAGENES_T)==32
muestra_img=IMAGENES_T[0]
with Image.open(RAIZ/muestra_img["original_path"]) as im: original_vista=im.convert("RGB").copy()
with Image.open(RAIZ/muestra_img["derived_path"]) as im: derivada_vista=im.copy()
fig,ax=plt.subplots(1,3,figsize=(12,3.5))
ax[0].imshow(original_vista);ax[0].set_title("RAW preservado");ax[0].axis("off")
ax[1].imshow(derivada_vista,cmap="gray" if derivada_vista.mode=="L" else None);ax[1].set_title("Derivado "+muestra_img["profile"]);ax[1].axis("off")
ax[2].bar(range(8),muestra_img["metrics"]["histogram8"]);ax[2].set_title("8 grupos de luminancia")
plt.tight_layout();plt.show()
print("HISTOGRAMA OK:",sum(muestra_img["metrics"]["histogram8"]),"píxeles medidos")
print("CRÉDITO ORIGINAL:",muestra_img["credit"],"· artículo:",muestra_img["article_id"])


## 5A · Lee el audio completo por bloques: código ejecutable

**¿Por qué?** Laura puede recibir WAV que duran varios minutos. El estudiante necesita ver que `sf.SoundFile` obtiene `samplerate` y `frames`; `blocks` permite avanzar sin cargarlo todo en RAM. En esta celda calcularemos el RMS global por acumulación y comprobaremos todas las muestras.

**HAZ ESTO AHORA:** ejecuta primero E · Audios. Esta práctica toma la primera voz real; puedes cambiarla por otra de las diez para observar duraciones distintas. Después compara el RMS con el derivado del paso 5. La conversión PCM16 real con remuestreo y escritura incremental está en `core.transformar_audios`: usa la celda **5B** para inspeccionar su código completo y ejecutarlo con tu propio WAV de más de 60 s.

**Esperas ver:** `AUDIO COMPLETO POR BLOQUES` y `muestras procesadas = muestras originales`. No necesitas escuchar el archivo entero para saber que se leyó íntegramente.

**Cómo se lee:** bloque es una parte consecutiva; frames es el número total de muestras temporales por canal. **Qué nos dice:** procesamiento íntegro y RMS calculado. **Qué NO permite concluir:** transcripción ni velocidad de red. **Qué error común:** usar solo los diez primeros segundos y declarar terminado todo el archivo.


In [ ]:
# 5A · CÓDIGO VISIBLE: lectura real incremental (sin cortar tras 60 segundos)
import numpy as np
import soundfile as sf

ruta_audio = RAIZ / AUDIOS_RAW[0]["original_path"]
suma_cuadrados, muestras, bloques = 0.0, 0, 0
with sf.SoundFile(ruta_audio) as archivo:
    tasa, total = archivo.samplerate, archivo.frames
    # Lee 10 segundos por bloque hasta agotar el archivo completo.
    for bloque in archivo.blocks(blocksize=tasa*10, dtype="float32", always_2d=True):
        mono = bloque.mean(axis=1)  # todos los canales → mono
        suma_cuadrados += float(np.sum(mono.astype(np.float64)**2))
        muestras += len(mono)
        bloques += 1
        print(f"Bloque {bloques}: {muestras} / {total} muestras")
assert muestras == total
rms_global = (suma_cuadrados / muestras)**0.5
print("AUDIO COMPLETO POR BLOQUES:", bloques, "bloques;",
      "duración:", round(muestras/tasa,2), "s;", "RMS:", round(rms_global,6))


## 5 · T: de WAV fuente a PCM16 mono 16 kHz

**HAZ ESTO AHORA.** Convierte las diez voces con `soundfile` (decodificación) y `scipy.signal.resample_poly` (remuestreo); aplica la ganancia elegida, guarda WAV PCM16 mono 16 kHz y calcula duración, RMS, cruces por cero, centroide espectral y recortes.

**Mini ficha:** `transformar_audios(raiz, entradas, ganancia, tasa_final)` devuelve un archivo por voz y sus métricas. Cambiar ganancia de 0,5 a 1,5 puede elevar RMS pero también causar saturación; escucha ambas muestras. La referencia humana de MINDS-14 no es una transcripción generada en este laboratorio.

**Cómo se lee:** RMS es amplitud eficaz, centroide es promedio ponderado de frecuencias, ZCR es proporción de cambios de signo. **Qué nos dice:** diferencias acústicas objetivas. **Qué NO permite concluir:** identidad, intención, emoción ni palabras del hablante. **Error común:** confundir 16 kHz de muestreo con 16 kHz de frecuencia predominante.
**Tu primer código:** ejecuta 5A para ver un ciclo de lectura de audio real antes de usar el motor de procesamiento masivo. **Tu reto de duración:** ejecuta 5B con WAV sintético de 65 s o tu archivo autorizado; no debes llevar archivos para completar el recorrido base.


In [ ]:
#@title 5 · T: transformar diez audios y observar la voz elegida
AUDIOS_T = core.transformar_audios(RAIZ,AUDIOS_RAW,ganancia=GANANCIA_AUDIO,tasa_final=16000)
assert len(AUDIOS_T)==10
tabla_audio=pd.DataFrame([{**{"source_record_id":x["source_record_id"]},**x["metrics"]} for x in AUDIOS_T])
display(tabla_audio[["source_record_id","duration_seconds","rms","zero_crossing_rate","spectral_centroid_hz","clipped_samples"]])
voz=AUDIOS_T[POSICION_VOZ-1]
with wave.open(str(RAIZ/voz["derived_path"]),"rb") as f:
    assert f.getnchannels()==1 and f.getsampwidth()==2 and f.getframerate()==16000
    audio_escucha=np.frombuffer(f.readframes(f.getnframes()),dtype="<i2").astype(np.float32)/32768
display(Audio(audio_escucha,rate=16000))
fig,ax=plt.subplots(1,2,figsize=(12,3))
ax[0].plot(np.arange(min(len(audio_escucha),32000))/16000,audio_escucha[:32000],lw=.6)
ax[0].set_title("Onda de la voz elegida");ax[0].set_xlabel("Segundos");ax[0].set_ylabel("Amplitud")
ax[1].specgram(audio_escucha[:min(len(audio_escucha),160000)],NFFT=512,Fs=16000,noverlap=384)
ax[1].set_title("Espectrograma (tiempo × frecuencia)");ax[1].set_xlabel("Segundos");ax[1].set_ylabel("Hz")
plt.tight_layout();plt.show()
print("VOZ ELEGIDA:",POSICION_VOZ,"· RMS:",voz["metrics"]["rms"],"· recortes:",voz["metrics"]["clipped_samples"])


## 5B · Prueba un WAV largo (como 3530042.wav) sin límite de un minuto

**¿Por qué?** El navegador rechazaba grabaciones largas. Laura no puede perder parte del audio para hacer que una demo funcione. El motor S11 **decodifica por bloques y guarda cada bloque PCM16**, con solapamiento de muestreo (*halo*) para evitar discontinuidades en los límites.

**HAZ ESTO AHORA:** elige `PROBAR_WAV_PROPIO`. Déjalo en `False` si no tienes un archivo: se creará un WAV **sintético de 65 segundos para prueba técnica**, claramente identificado y separado del corpus real. Si deseas probar `3530042.wav`, súbelo voluntariamente a Colab en el control de archivos o deja el WAV en la ruta señalada de VS Code. No descargamos contenido editorial de terceros automáticamente.

**Código que verás:** creación del WAV de prueba, inspección de `sf.SoundFile`, llamado a `core.transformar_audios(..., block_seconds=8, on_block=...)` y comprobación de duración y SHA. El estudiante puede revisar la implementación completa con `inspect.getsource` (es código real, no pseudocódigo).

**Esperas ver:** `AUDIO LARGO COMPLETO`, al menos 2 bloques en la prueba sintética, duración ≥65 s, SHA de origen idéntico y derivado PCM16.

**Cómo se lee:** `processed_input_frames/total_input_frames` avanza hasta 1. **Qué nos dice:** que no se descartó el resto del archivo. **Qué NO permite concluir:** que Web Audio del navegador pueda decodificar todos los códecs o tamaños sin límite de RAM. **Qué error común:** declarar que «el audio es de un minuto» porque solo se visualizó una ventana.


In [ ]:
# 5B · Código y prueba de audio >60 s (opcional subir WAV propio)
PROBAR_WAV_PROPIO = False #@param {type:"boolean"}
MOSTRAR_CODIGO_MOTOR = False #@param {type:"boolean"}
import io, hashlib, wave, inspect
import numpy as np
import soundfile as sf

if PROBAR_WAV_PROPIO:
    if EN_COLAB:
        from google.colab import files
        subidos = files.upload()  # solo cuando el estudiante lo elige
        if len(subidos) != 1: raise ValueError("Selecciona exactamente un WAV")
        nombre, contenido = next(iter(subidos.items()))
    else:
        nombre = "audio_para_probar.wav"
        ruta_local = Path.cwd() / nombre
        if not ruta_local.is_file(): raise FileNotFoundError(ruta_local)
        contenido = ruta_local.read_bytes()
else:
    # Prueba artificial para validar DURACIÓN, nunca se atribuye a una noticia.
    nombre = "ensayo_65_segundos.wav"
    fs_prueba = 16000
    periodo = np.arange(fs_prueba, dtype=np.float32)/fs_prueba
    un_segundo = np.rint(np.sin(2*np.pi*220*periodo)*12000).astype("<i2")
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as wav:
        wav.setnchannels(1); wav.setsampwidth(2); wav.setframerate(fs_prueba)
        for segundo in range(65): wav.writeframesraw(un_segundo.tobytes())
    contenido = buffer.getvalue()

destino_raw = RAIZ / "raw" / "audios_alumno" / nombre.split("/")[-1].split("\\")[-1]
hash_original = hashlib.sha256(contenido).hexdigest()
core.guardar_sin_sobrescribir(destino_raw, contenido, hash_original)
with sf.SoundFile(destino_raw) as wav_original:
    duracion_original = wav_original.frames / wav_original.samplerate
    total_frames = wav_original.frames
print("DURACIÓN RECIBIDA:", round(duracion_original,2), "segundos · sin recorte")
entrada = {
    "original_path": destino_raw.relative_to(RAIZ).as_posix(),
    "original_sha256": hash_original,
    "source_id": hash_original[:12],
    "source_kind": "uploaded_by_student" if PROBAR_WAV_PROPIO else "synthetic_long_audio_test",
    "article_id": None, "source_record_id": None,
    "source_url": None, "license": "ejercicio_local_no_publicar",
}
def mostrar_avance(info):
    if info["blocks"]==1 or info["progress"]==1 or info["blocks"]%3==0:
        print("BLOQUE",info["blocks"],"·",info["processed_input_frames"],"/",
              info["total_input_frames"],"muestras ·",round(info["progress"]*100,1),"%")
resultado_largo = core.transformar_audios(
    RAIZ,[entrada],ganancia=1.0,block_seconds=8,on_block=mostrar_avance)
metrica_larga = resultado_largo[0]["metrics"]
assert metrica_larga["processed_input_frames"] == total_frames
assert abs(metrica_larga["duration_seconds"]-duracion_original) < 0.02
assert core.sha256(destino_raw) == hash_original
assert core.sha256(RAIZ/resultado_largo[0]["derived_path"]) == resultado_largo[0]["derived_sha256"]
print("AUDIO LARGO COMPLETO:",round(metrica_larga["duration_seconds"],2),
      "segundos ·",metrica_larga["processing_blocks"],"bloques · SHA OK")
if MOSTRAR_CODIGO_MOTOR:
    print(inspect.getsource(core.transformar_audios))  # implementación real completa


## 6 · L: catálogo único, procedencias distintas

**Problema de Laura:** no queremos 42 archivos sueltos. Queremos un catálogo consultable con modalidad, hashes, perfil, métricas, identificador de origen y ruta para reabrir cada derivado.

**HAZ ESTO AHORA.** Ejecuta la carga. Se crean `outputs/catalogo.jsonl`, `outputs/catalogo.parquet`, `outputs/decision.json` y `outputs/resumen.json`; se reabre el Parquet y se produce `manifest_s11.json`. Los datos del usuario se limitan a elecciones estructuradas.

**Salida:** `L · CATÁLOGO: 42 filas (32 imágenes + 10 audios)` y `MANIFIESTO OK`.

**Cómo se lee:** una fila representa un derivado de un RAW; los activos se consultan por `modality` y `source_id`. **Qué nos dice:** es posible recuperar origen y transformación. **Qué NO permite concluir:** que dos fuentes distintas sean una única evidencia editorial. **Error común:** unir por una clave de origen inexistente y atribuir audios MINDS-14 a artículos.
**Antes de continuar:** la práctica **6B** enseñará JSON vs JSONL vs Parquet con el mismo registro; la práctica **6C** comprobará que Laura puede recuperar un archivo usando su ruta en el catálogo. Ambas se ejecutan después de 6 y antes de 7.


In [ ]:
#@title 6 · L: JSONL, Parquet, decisiones y manifiesto
CATALOGO=core.cargar_catalogo(RAIZ,IMAGENES_T,AUDIOS_T,DECISIONES)
MANIFIESTO=core.crear_manifiesto(RAIZ,CATALOGO["filas"])
INFORME=core.verificar(RAIZ)
assert INFORME["ok"] and (INFORME["imagenes"],INFORME["audios"])==(32,10),INFORME
print("MANIFIESTO OK:",INFORME)
display(CATALOGO["tabla"][["modality","source_kind","source_id","profile","metric_name","metric_value"]].head(8))
print("CROSS-DATASET:",CATALOGO["resumen"]["fuentes_cruzadas"],"vínculos inventados")


## 6B · De una lista de Python a JSON, JSONL y Parquet: ¿cuál elegir y por qué?

Laura tiene que entregar un **catálogo que pueda consultar otro equipo**. Tres formatos pueden representar los mismos datos, pero no se usan de la misma manera.

**El mismo ejemplo, representado de tres formas:**

**JSON (lista de dos objetos, un documento completo):**
```json
[
  {"asset_id":"img01","modality":"image","status":"ok"},
  {"asset_id":"aud01","modality":"audio","status":"ok"}
]
```

**JSONL (dos líneas independientes, sin corchetes ni comas entre líneas):**
```jsonl
{"asset_id":"img01","modality":"image","status":"ok"}
{"asset_id":"aud01","modality":"audio","status":"ok"}
```

**Parquet:** contiene las mismas tres columnas y sus tipos en un archivo **binario por columnas**; se consulta con `pandas.read_parquet`, no con un editor de texto. Puede reducir lectura y almacenamiento, pero en dos registros mínimos no tiene por qué ser más pequeño que JSONL.

**¿Qué implementarás?** Ejecuta el ejemplo de dos filas, crea tres archivos, vuelve a leerlos y verifica igualdad. Luego abre el catálogo **real** de Laura que acabas de generar en el paso 6. Filtra audios e imágenes y descubre por qué una tabla facilita consultar ambos sin atribuirles una relación.

**Salida esperada:** `FORMATOS REABIERTOS: 2 filas equivalentes` y `CATÁLOGO REAL REABIERTO:` seguido de 42 filas. El tamaño en bytes se mide: no se supone cuál es menor.

**Cómo se lee:** JSON es un documento completo; JSONL una secuencia de registros JSON; Parquet una tabla tipada. **Qué nos dice:** representan la misma información lógica en este ejemplo. **Qué NO permite concluir todavía:** cuál será más rápido para millones de registros sin hacer un benchmark comparable. **Qué error común:** abrir Parquet con `read_text()` o creer que JSONL es un archivo JSON con un gran arreglo.


In [ ]:
# 6B · EXPERIMENTO EJECUTABLE: tres formatos de la MISMA tabla
# Tareas resueltas: crear, escribir y reabrir. Cambia las filas y repite.
import json
import pandas as pd

filas_ejemplo = [
    {"asset_id": "img01", "modality": "image", "status": "ok"},
    {"asset_id": "aud01", "modality": "audio", "status": "ok"},
]
carpeta_formatos = RAIZ / "outputs" / "formatos_practica"
carpeta_formatos.mkdir(parents=True, exist_ok=True)

# JSON: un documento completo con una lista de objetos.
ruta_json = carpeta_formatos / "ejemplo.json"
ruta_json.write_text(json.dumps(filas_ejemplo, indent=2), encoding="utf-8")

# JSONL: cada línea es un objeto JSON completo. Ideal para registros incrementales.
ruta_jsonl = carpeta_formatos / "ejemplo.jsonl"
with ruta_jsonl.open("w", encoding="utf-8") as salida:
    for fila in filas_ejemplo:
        salida.write(json.dumps(fila) + "\n")

# Parquet: columnas tipadas y almacenamiento binario.
ruta_parquet = carpeta_formatos / "ejemplo.parquet"
pd.DataFrame(filas_ejemplo).to_parquet(ruta_parquet, index=False)

# Reabrimos los tres formatos de manera adecuada, NO como una sola cadena.
desde_json = json.loads(ruta_json.read_text(encoding="utf-8"))
desde_jsonl = [json.loads(linea) for linea in ruta_jsonl.read_text(encoding="utf-8").splitlines()]
desde_parquet = pd.read_parquet(ruta_parquet).to_dict("records")

assert desde_json == desde_jsonl == desde_parquet
print("FORMATOS REABIERTOS:", len(desde_json), "filas equivalentes")
for archivo in (ruta_json, ruta_jsonl, ruta_parquet):
    print(archivo.name, "→", archivo.stat().st_size, "bytes")

# Ahora usa los archivos del ETL REAL de Laura, no el ejemplo artificial.
real_jsonl = RAIZ / "outputs" / "catalogo.jsonl"
real_parquet = RAIZ / "outputs" / "catalogo.parquet"
assert real_jsonl.exists() and real_parquet.exists(), "Ejecuta primero la etapa 6 · L"
catalogo_real = pd.read_parquet(real_parquet)
assert len(catalogo_real) == len(real_jsonl.read_text(encoding="utf-8").splitlines())
print("CATÁLOGO REAL REABIERTO:", len(catalogo_real), "filas")
display(catalogo_real.groupby("modality").size().rename("registros"))
# Modifica aquí el filtro a "image" y observa las rutas disponibles.
display(catalogo_real[catalogo_real["modality"] == "audio"][["source_kind","original_path","derived_path"]].head())


## 6C · ¿Cómo recuperaría Laura un activo mañana sin ejecutar la transformación?

Un catálogo no es una carpeta de fotos: es **un índice de direcciones y metadatos**. Cada fila del Parquet apunta con `original_path` a su RAW y con `derived_path` al resultado. `RAIZ / ruta` reconstruye la ubicación.

**HAZ ESTO AHORA:** ejecuta la celda para escoger automáticamente una fila de imagen y otra de audio, encontrar ambos archivos y comprobar que sus bytes coinciden con sus SHA-256. No tendrás que recordar nombres.

**Salida esperada:** `RECUPERACIÓN OK: 2 originales y 2 derivados comprobados`.

**Cómo se lee:** dos filas de fuentes distintas recuperadas por ruta. **Qué nos dice:** el producto funciona sin repetir E/T. **Qué NO permite concluir:** autenticidad editorial o semántica de la voz. **Qué error común:** guardar rutas absolutas del computador de alguien en el catálogo portable.


In [ ]:
# 6C · RECUPERAR sin reprocesar: se consultan rutas, no se vuelve a ejecutar el ETL
import hashlib

catalogo = pd.read_parquet(RAIZ / "outputs" / "catalogo.parquet")
filas_recuperadas = []
for modalidad in ("image", "audio"):
    fila = catalogo.loc[catalogo["modality"] == modalidad].iloc[0]
    for campo_ruta, campo_sha in (
        ("original_path", "original_sha256"),
        ("derived_path", "derived_sha256"),
    ):
        archivo = RAIZ / fila[campo_ruta]
        assert archivo.is_file(), f"No existe: {archivo}"
        huella = hashlib.sha256(archivo.read_bytes()).hexdigest()
        assert huella == fila[campo_sha], f"Contenido alterado: {archivo}"
    filas_recuperadas.append(fila)
print("RECUPERACIÓN OK: 2 originales y 2 derivados comprobados")
display(pd.DataFrame(filas_recuperadas)[["modality","source_id","profile","original_path","derived_path"]])


## 7 · Calidad: repetir, recuperar y detectar un error

**HAZ ESTO AHORA.** Ejecuta otra vez la extracción y transformación con los mismos parámetros. Los SHA originales y derivados deben ser idénticos. Después inyecta bytes dañados sobre una COPIA TEMPORAL del expediente: el verificador debe rechazarla. No modifiques los RAW reales.

**Salida:** `IDEMPOTENCIA OK` y `QA · CORRUPCIÓN DETECTADA EN COPIA`.

**Cómo se lee:** la prueba positiva compara SHA antes/después; la negativa verifica que un cambio no autorizado se detecta. **Qué nos dice:** el proceso no duplica ni ignora alteraciones. **Qué NO permite concluir:** que la fuente informativa dice la verdad o que el pipeline ya puede procesar millones de archivos. **Error común:** asumir que la coincidencia de hashes reemplaza una evaluación de calidad semántica.


In [ ]:
#@title 7 · Reejecución idempotente y prueba negativa segura
otra_img=core.extraer_imagenes(RAIZ,max_imagenes=32)
otra_aud=core.extraer_audios(RAIZ,max_audios=10)
otra_img_t=core.transformar_imagenes(RAIZ,otra_img,lado=LADO_IMAGEN,perfil=PERFIL_IMAGEN)
otra_aud_t=core.transformar_audios(RAIZ,otra_aud,ganancia=GANANCIA_AUDIO)
assert [x["derived_sha256"] for x in otra_img_t]==[x["derived_sha256"] for x in IMAGENES_T]
assert [x["derived_sha256"] for x in otra_aud_t]==[x["derived_sha256"] for x in AUDIOS_T]
print("IDEMPOTENCIA OK: 32 imágenes + 10 audios reproducidos sin cambios")
PRUEBA_FALLO=core.probar_detencion(RAIZ)
assert PRUEBA_FALLO=="CORRUPCION_DETECTADA" and core.verificar(RAIZ)["ok"]


## 8 · ¿Por qué esto forma parte de Big Data?

**La variedad ya es real:** procesamos bytes de imagen (WEBP→PNG) y bytes de audio (WAV μ-law/PCM→PCM16) con transformaciones y metadatos diferentes bajo el mismo contrato. **El volumen de 42 activos NO demuestra una necesidad de distribución.** El lote nos deja medir tamaños y proyectar, sin vender una extrapolación como benchmark.

**HAZ ESTO AHORA.** Calcula bytes RAW y derivados reales y compara tamaños; si tu proyecto escala a un millón de activos, anota que requerirá particionado, paralelismo, cuotas y monitoreo. Eso será una hipótesis a comprobar con Dask/Spark, no una conclusión de este notebook.

**Cómo se lee:** el volumen acumulado es suma de tamaños; una estimación lineal supone tamaños iguales. **Qué nos dice:** dimensiones del problema de almacenamiento. **Qué NO permite concluir:** velocidad de un clúster ni tiempos reales a escala. **Error común:** extrapolar 42 observaciones como si todas las futuras fuentes tuvieran igual distribución.


In [ ]:
#@title 8 · Volumen medido y proyección hipotética, no benchmark
filas=CATALOGO["filas"]
bytes_raw=sum((RAIZ/x["original_path"]).stat().st_size for x in filas)
bytes_derived=sum((RAIZ/x["derived_path"]).stat().st_size for x in filas)
print("DATOS MEDIDOS:",len(filas),"activos; RAW",bytes_raw,"bytes; derivados",bytes_derived,"bytes")
estimacion_millones=bytes_raw/len(filas)*1_000_000
print("PROYECCIÓN LINEAL DE RAW PARA 1 MILLÓN (NO MEDICIÓN):",round(estimacion_millones/1e9,2),"GB")
assert bytes_raw>0 and bytes_derived>0


## 8B · El tamaño del lote sí cambia: trabaja con **cientos o miles** de archivos

Laura ya produjo 42 derivados en la ruta básica. **El nuevo problema** no es qué es un píxel ni volver a instalar MongoDB: es ¿cómo hacer el **mismo ETL** cuando llegan más fuentes y el equipo tiene memoria limitada?

**Las tres colecciones se mantienen separadas:**
- **32 fotografías** periodísticas auténticas del inventario previamente autorizado del curso. Tienen `article_id`.
- **40, 200 o 1.000 imágenes distintas** de escritura manuscrita normalizadas a **8×8** desde el corpus de reconocimiento óptico UCI, distribuido por `scikit-learn.load_digits`. Son muestras derivadas de matrices originales, **no fotografías de contratos**. Fuente: Alpaydin y Kaynak (1998), UCI, DOI 10.24432/C50P49, CC BY 4.0.
- **30, 100 o 400 voces humanas distintas** del corpus autorizado MINDS-14 español, CC BY 4.0. Las diez voces iniciales se conservan, y las adicionales se seleccionan sin duplicar los bytes.

| Nivel que puedes ejecutar | Imágenes originales: prensa + manuscritos | Audios reales | RAW distintos | Derivados: 2 perfiles por imagen + 1 por voz |
|---|---:|---:|---:|---:|
| **Ensayo guiado** | 32 + 40 | 30 | 102 | **174** |
| **Nivel medio** | 32 + 200 | 100 | 332 | **564** |
| **Reto de escala** | 32 + 1.000 | 400 | 1.432 | **2.464** |

**OJO:** 2.464 derivados no significan 2.464 fotografías distintas, ni gigabytes garantizados. Los 32 archivos de prensa siguen siendo 32 originales, y los dígitos 8×8 tienen pocos bytes. **Big Data se aprende midiendo volumen real, particiones, costo de transformación y límites; el clúster aparece después**.

**HAZ ESTO AHORA.** En la siguiente celda deja `ensayo` para empezar. Luego cambia a `medio` y compara. `reto` es profundización **opcional**, no debe impedir que termines el expediente de 42 filas. Elige `TAMANO_LOTE` = 8, 32 o 64 y observa cuántas particiones y cuántos registros por segundo produjo **tu propia ejecución**. Las huellas del expediente oficial no se alteran.

**Salida exacta:** `RETO DE ESCALA`, líneas `PARTICIÓN 01` y `ESCALA VERIFICADA`. **Si falla:** disminuye nivel y lote; revisa si el equipo tiene memoria o disco suficiente y que el runtime conserva la descarga de MINDS-14. Se trabaja **por fragmentos** y cada fragmento se escribe y se vuelve a leer desde Parquet.

**Cómo se lee:** el tiempo y la tasa corresponden a cada partición. **Qué nos dice:** cómo se comportó **este equipo y este lote**. **Qué NO permite concluir todavía:** que una máquina distribuida sea más rápida o que el rendimiento escale linealmente. **Error común:** comparar una primera ejecución fría con otra que reutiliza caché y atribuir la diferencia al tamaño del lote.

**Memoria:** la tabla incluye `peak_process_rss_mb`, el **máximo de memoria residente del proceso hasta ese momento**, no la RAM exclusiva de una partición. Sirve para detectar presión de memoria, pero la caché del runtime y tareas anteriores afectan esa cifra. Nunca la interpretes como memoria distribuida entre workers.


In [ ]:
#@title 8B · Ejecutar un lote cada vez mayor y medir sus particiones
MODO_ESCALA = "ensayo" #@param ["ensayo","medio","reto"]
TAMANO_LOTE = 32 #@param [8,16,32,64,128] {type:"raw"}
ACTIVAR_ESCALA = True #@param {type:"boolean"}
if ACTIVAR_ESCALA:
    resultado_escala=core.ejecutar_escala_s11(RAIZ,modo=MODO_ESCALA,tamano_lote=TAMANO_LOTE)
    estadisticas_lotes=resultado_escala["particiones"]
    verificacion_escala=resultado_escala["verified"]
    assert verificacion_escala["ok"]
    print("ESCALA VERIFICADA:",verificacion_escala["rows"],"derivados;",
          verificacion_escala["sources"],"fuentes distintas;",
          verificacion_escala["partitions"],"particiones")
    print("PROCESAMIENTO LOCAL: 1 worker; sin Spark ni Dask")
    display(estadisticas_lotes[["name","rows","original_sources","seconds","rows_per_second","bytes_raw_referenced","bytes_derived","peak_process_rss_mb"]])
else:
    print("Escala desactivada: el expediente oficial de 42 filas sigue completo.")


## 8C · Interpreta tus propios números y descubre el límite real

**HAZ ESTO AHORA:** mira las particiones y registra el perfil elegido y sus resultados en el archivo de métricas; cambia `TAMANO_LOTE`, ejecuta **8B** nuevamente y vuelve a ejecutar esta celda. No se piden respuestas abiertas. Una buena decisión se basa en las tasas, tamaños y fallos que **tú** observaste.

**Cuatro preguntas sobre la ejecución:**
1. ¿Qué aumentó: **fuentes únicas**, **derivados** o ambos?
2. ¿Por qué 2.464 derivados de 8×8 podrían ocupar menos que unos pocos audios?
3. ¿Qué diferencia hay entre 8 y 64 fuentes por partición, para este equipo y estas entradas?
4. ¿Qué prueba falta para afirmar que un clúster mejora el rendimiento? **Ejecutar el mismo trabajo con workers y controlar caché, hardware, I/O y resultado**.

**Cómo se lee:** el archivo `comparaciones_s11.csv` guarda métricas de tus pruebas. **Qué nos dice:** conteos, segundos y throughput medidos, no predicciones. **Qué NO permite concluir todavía:** escalamiento de un sistema distribuido ni rendimiento de millones de activos. **Error común:** sumar bytes RAW por cada derivado; cuando un original genera dos derivados, **su RAW se cuenta una sola vez**.

**Conserva lo producido:** la celda crea `reporte_...zip` con manifiesto, particiones Parquet y tabla de tiempos. Para descargarlo en Colab activa `DESCARGAR_REPORTE_ESCALA` dentro de la celda. **Este ZIP ligero no incluye medios editoriales ni audios:** está diseñado para registrar las mediciones y reconstruir el procesamiento con sus fuentes autorizadas. Si deseas conservar también todos los derivados, guarda la carpeta de trabajo por separado y revisa primero derechos y espacio disponible.

**Gráfico de rendimiento:** la barra de cada partición representa `rows_per_second` medidos. Si una barra es diferente, investiga complejidad de las imágenes, duración de los audios, caché y tamaño del lote antes de afirmar que una configuración es superior.


In [ ]:
#@title 8C · Registrar el experimento y verificar que RAW no se cuente dos veces
if ACTIVAR_ESCALA:
    resumen=resultado_escala["manifest"]
    filas_reales=int(estadisticas_lotes["rows"].sum())
    segundos=float(estadisticas_lotes["seconds"].sum())
    raw_unicos=sum((RAIZ/ruta).stat().st_size for ruta in resumen["originals"])
    bytes_derivados=int(estadisticas_lotes["bytes_derived"].sum())
    assert filas_reales==resumen["expected_derivatives"]
    assert len(resumen["originals"])==resumen["total_unique_sources"]
    assert filas_reales>resumen["total_unique_sources"]
    assert core.verificar_lotes_escala(RAIZ,resultado_escala["manifest_path"])["ok"]
    informe={
        "modo":MODO_ESCALA,"tamano_lote":TAMANO_LOTE,
        "fuentes_unicas":resumen["total_unique_sources"],
        "derivados":filas_reales,"particiones":len(estadisticas_lotes),
        "segundos_transformacion_y_parquet":round(segundos,3),
        "derivados_por_segundo":round(filas_reales/max(segundos,0.001),2),
        "raw_bytes_sin_duplicar":int(raw_unicos),
        "derived_bytes":bytes_derivados,"workers":1,
        "peak_process_rss_mb":float(estadisticas_lotes["peak_process_rss_mb"].max())}
    destino_comparaciones=RAIZ/"outputs"/"escala"/"comparaciones_s11.csv"
    actuales=pd.read_csv(destino_comparaciones) if destino_comparaciones.exists() else pd.DataFrame()
    tablero=pd.concat([actuales,pd.DataFrame([informe])],ignore_index=True).drop_duplicates(
        subset=["modo","tamano_lote"],keep="last")
    tablero.to_csv(destino_comparaciones,index=False)
    print("COMPARACIÓN REAL:",informe)
    print("EXPERIMENTOS GUARDADOS:",len(tablero),"·",destino_comparaciones)
    display(tablero)
    # Diagrama construido EXCLUSIVAMENTE con datos medidos por tus propias particiones.
    plt.figure(figsize=(9,3))
    plt.bar(estadisticas_lotes["name"], estadisticas_lotes["rows_per_second"])
    plt.title("Rendimiento medido por partición · "+MODO_ESCALA)
    plt.ylabel("Derivados por segundo")
    plt.xlabel("Partición")
    plt.xticks(rotation=45,ha="right")
    plt.tight_layout()
    plt.show()
    # El reporte de escala guarda catálogo y mediciones, NO redistribuye fotografías editoriales.
    import zipfile
    REPORTE_ESCALA=RAIZ/"outputs"/"escala"/f"reporte_{MODO_ESCALA}_b{TAMANO_LOTE}.zip"
    with zipfile.ZipFile(REPORTE_ESCALA,"w",compression=zipfile.ZIP_DEFLATED) as paquete:
        paquete.write(resultado_escala["manifest_path"],"escala_manifest.json")
        for part in resumen["partitions"]:
            ruta_part=resultado_escala["manifest_path"].parent/part["name"]
            paquete.write(ruta_part,"particiones/"+part["name"])
        paquete.write(destino_comparaciones,"comparaciones_s11.csv")
        paquete.writestr("LEEME.txt",
            "Informe académico de métricas S11. Incluye manifiesto, particiones Parquet y comparaciones. "
            "NO incluye archivos multimedia RAW ni derivados; para reconstruirlos ejecuta el cuaderno "
            "S11 con las fuentes UCI, el snapshot editorial autorizado y MINDS-14. "
            "No se atribuyen audios a noticias o contratos.\n")
    assert zipfile.ZipFile(REPORTE_ESCALA).testzip() is None
    print("INFORME DE ESCALA LISTO:",REPORTE_ESCALA,"·",REPORTE_ESCALA.stat().st_size,"bytes")
    DESCARGAR_REPORTE_ESCALA=False #@param {type:"boolean"}
    if DESCARGAR_REPORTE_ESCALA and EN_COLAB:
        from google.colab import files as archivos_colab
        archivos_colab.download(str(REPORTE_ESCALA))
else:
    print("La comparación espera que actives primero la celda 8B.")


## 8D · ¿Qué pasa MIENTRAS se ejecuta? Nuestro monitor E2E real

**El problema de Laura:** ya tenemos un ETL correcto, pero con 102 fuentes no sabemos cuál está trabajando, cuál terminó ni qué archivos fallaron hasta leer el resumen. Un proceso visible ayuda a decidir, no reemplaza la comprobación.

**La práctica visual está dentro de la [presentación S11](https://jazaineam1.github.io/BigData2026/Presentaciones/s11-etl-audio-imagenes.html#s18)**: utiliza un formulario local para subir hasta seis archivos, observa original/derivado, progreso por archivo y ZIP. Este cuaderno es la segunda parte de la **misma sesión**: usa imágenes y audios reales del curso, no la demostración sintética de la página.

**Mini diccionario:**
- **Thread (hilo):** secuencia de ejecución dentro de un mismo proceso. Comparte RAM; no equivale a un nodo distribuido. Python puede limitar el código CPU puro por el GIL; NumPy/SciPy/Pillow pueden liberar el GIL en operaciones nativas.
- **Future:** tarea presentada a un ejecutor; el coordinador conoce cuándo termina.
- **Cola acotada:** máximo `2 × trabajadores` tareas enviadas al mismo tiempo. Evita una acumulación de miles de objetos.
- **Evento:** mensaje emitido por una acción real: `queued`, `processed`, `partition_saved`, `verified`, `retry` o `cancelled`.
- **Throughput:** derivados terminados por segundo medidos, no predichos.
- **Cancelación cooperativa:** deja concluir lo que ya se está ejecutando y evita presentar un lote parcial como completo.

**HAZ ESTO AHORA:** ejecuta 8D para cargar el motor y el panel. Después abre la E2E en la presentación para transformar una imagen y un audio. La página no accede al kernel de Colab y no controla los threads Python: cada monitor enseña los eventos de **su propio entorno**.

**Salida esperada:** `MOTOR LIVE LISTO`. Si falla la instalación de widgets, la ejecución conserva `events.jsonl` verificable y mensajes en consola.

**Cómo se lee:** la carga del motor no significa que ya terminó un ETL. **Qué nos dice:** podemos instrumentar el procesamiento. **Qué NO permite concluir todavía:** que cuatro hilos sean más rápidos que uno. **Error común:** interpretar un contador animado como una medición real.


In [ ]:
#@title 8D · Motor de threads y tablero observable sin cuentas nuevas
import importlib.util as spec_util
from urllib.request import urlopen, Request
def cargar_modulo_s11(nombre):
    opciones=[Path("utils")/(nombre+".py"),Path("../utils")/(nombre+".py")]
    ruta=next((p for p in opciones if p.is_file()),None)
    if ruta is None:
        ruta=RAIZ/(nombre+".py")
        if not ruta.exists():
            url=f"https://raw.githubusercontent.com/jazaineam1/BigData2026/main/utils/{nombre}.py"
            with urlopen(Request(url,headers={"User-Agent":"BigData-UCentral-S11/1.0"}),timeout=35) as respuesta:
                ruta.write_bytes(respuesta.read())
    especificacion=spec_util.spec_from_file_location(nombre,ruta)
    modulo=spec_util.module_from_spec(especificacion)
    especificacion.loader.exec_module(modulo)
    return modulo
motor_live=cargar_modulo_s11("s11_etl_live")
panel_mod=cargar_modulo_s11("s11_live_dashboard")
assert motor_live.SCHEMA=="s11-live-threads/v1"
print("MOTOR LIVE LISTO · estados medidos, workers 1/2/4, archivos verificados")


## 8E · Haz que el trabajo avance EN VIVO: dos hilos y una cola

**HAZ ESTO AHORA:** elige `base` (32 fotografías + 10 audios), `ensayo` (102 fuentes) o `medio` (332 fuentes, opcional por memoria y tiempo) y ejecuta **8E-A**. Se abrirá un panel con porcentaje, contadores, fuente, etapa, derivados por segundo, tareas en vuelo y errores. **Antes de ejecutar 8E-B**, observa el avance y prueba **Pausar admisión** y **Reanudar**; los trabajos que ya estaban activos pueden terminar. También puedes cancelar; el expediente parcial no se marcará como válido.

**Después ejecuta 8E-B** para comprobar el manifiesto. Si cancelaste, vuelve a ejecutar 8E-A para empezar de nuevo: no borres los RAW ni edites las huellas manualmente.

**Salida esperada:** `LIVE VERIFICADO` y una cantidad acorde con el tamaño seleccionado. Si la interfaz no admite widgets, `events.jsonl` siempre conserva los eventos reales.

**Cómo se lee:** `procesados/total` proviene de tareas terminadas; los eventos de Parquet aparecen cuando el coordinador escribe la partición. **Qué nos dice:** cuál tarea avanzó y cuánto se midió. **Qué NO permite concluir:** que tener dos hilos produzca el doble de rendimiento. **Error común:** confundir `en vuelo` con hilos ejecutando CPU simultáneamente.

**En el tablero:** `MiB/s` registra bytes de originales más derivados completados por segundo; `RSS pico` es un máximo acumulado de memoria de todo Python, no memoria exclusiva de cada hilo.

**Ruta corta para la clase:** empieza con `base`, luego prueba `ensayo` si tienes tiempo. `medio` procesa 32 fotografías, 200 imágenes UCI y 100 voces y sirve como profundización; no es necesario para completar la autocomprobación de S11.


In [ ]:
#@title 8E-A · INICIAR: ETL en vivo con botones de pausar y cancelar
FUENTES_LIVE="base" #@param ["base","ensayo","medio"]
HILOS_LIVE=2 #@param [1,2,4] {type:"raw"}
LOTE_LIVE=16 #@param [8,16,32] {type:"raw"}
if FUENTES_LIVE in ("ensayo","medio"):
    IMAGENES_LIVE=IMAGENES_RAW+core.extraer_imagenes_digitos(RAIZ,40 if FUENTES_LIVE=="ensayo" else 200)
    AUDIOS_LIVE=core.extraer_audios(RAIZ,max_audios=30 if FUENTES_LIVE=="ensayo" else 100)
else:
    IMAGENES_LIVE=IMAGENES_RAW
    AUDIOS_LIVE=AUDIOS_RAW
PANEL_LIVE=panel_mod.LiveMonitor(len(IMAGENES_LIVE)+len(AUDIOS_LIVE),
                                title="Laura · ETL Python en ejecución REAL")
PANEL_LIVE.show()
FUTURO_LIVE=PANEL_LIVE.start(motor_live,RAIZ,IMAGENES_LIVE,AUDIOS_LIVE,
                             workers=HILOS_LIVE,batch_size=LOTE_LIVE,
                             label="clase",core=core)
print("TRABAJOS LANZADOS · controla la admisión en el panel antes de ejecutar 8E-B")


In [ ]:
#@title 8E-B · ESPERAR y verificar; cancelado NO significa completo
RESULTADO_LIVE=PANEL_LIVE.wait()
if RESULTADO_LIVE["status"]=="complete":
    assert RESULTADO_LIVE["verified"]["ok"]
    assert RESULTADO_LIVE["verified"]["rows"]==2*len(IMAGENES_LIVE)+len(AUDIOS_LIVE)
    assert motor_live.verify_run(RAIZ,RESULTADO_LIVE["manifest_path"],core=core)["ok"]
    print("LIVE VERIFICADO ·",RESULTADO_LIVE["verified"])
    print("EVENTOS PERSISTIDOS:",RESULTADO_LIVE["events_path"])
    print("HUELLA LÓGICA:",RESULTADO_LIVE["manifest"]["fingerprint"][:24])
else:
    print("EJECUCIÓN",RESULTADO_LIVE["status"].upper(),
          "· no se considera entrega completa. Reejecuta 8E-A para recuperarla.")


## 8E-C · Mira, ejecuta y comprende un ThreadPoolExecutor antes del benchmark

**El problema de Laura:** dos archivos no tienen que esperar uno detrás del otro para comenzar. **Thread** es un trabajador dentro de este proceso. Un **Future** es la promesa de un resultado. **submit** programa una función. **as_completed** entrega resultados en orden de finalización, no de entrada.

**HAZ ESTO AHORA:** usa una imagen y un audio que ya extrajiste. Ejecuta el código siguiente; después cambia `max_workers=2` a 1. Compara las salidas y los SHA, no solo el reloj. Para grandes lotes el motor real limita la cola y guarda manifiestos.

**Cómo se lee:** verás dos tareas finalizadas y sus hashes. **Qué nos dice:** dos trabajadores pueden procesar tareas independientes y conservar la integridad. **Qué NO permite concluir:** que haya dos núcleos ocupados todo el tiempo ni un speedup de 2×. **Qué error común:** confundir la secuencia de finalización con un orden de catálogo o suponer que dos hilos duplican memoria/hardware.


In [ ]:
# 8E-C · CÓDIGO VISIBLE: los dos archivos del caso, no tiempos inventados
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter

def procesar_una_fuente(modalidad, fuente):
    inicio = perf_counter()
    if modalidad == "image":
        filas = core.transformar_imagenes(RAIZ,[fuente],lado=160,perfil="grises",verbose=False)
    else:
        filas = core.transformar_audios(RAIZ,[fuente],ganancia=1.0,verbose=False)
    resultado = filas[0]
    ruta = RAIZ / resultado["derived_path"]
    assert core.sha256(ruta) == resultado["derived_sha256"]
    return modalidad, resultado["derived_sha256"], perf_counter()-inicio

trabajos = [("image",IMAGENES_RAW[0]),("audio",AUDIOS_RAW[0])]
# CAMBIA a 1 y vuelve a ejecutar; los SHA deben mantenerse idénticos.
with ThreadPoolExecutor(max_workers=2) as ejecutor:
    futuros = [ejecutor.submit(procesar_una_fuente,*item) for item in trabajos]
    completados = [futuro.result() for futuro in as_completed(futuros)]

assert {modalidad for modalidad,sha,secs in completados} == {"image","audio"}
for modalidad,sha,segundos in completados:
    print("THREAD TERMINÓ:", modalidad,"· SHA:",sha[:16],"· tiempo:",round(segundos,3),"s")
print("RESULTADOS REALES:",len(completados),"derivados verificados")


## 8F · ¿Cuántos threads realmente convienen?

La línea base ejecuta **el mismo trabajo con 1 thread**. La comparación utiliza **2 y 4 threads**. Cada configuración se repite **3 veces**, y se rota el orden para mitigar diferencias por caché.

**Conceptos para interpretar tus medidas:**
- `speedup = mediana(segundos con 1) / mediana(segundos con N)`. Un valor menor que 1 indica que **empeoró**, no que el experimento falló.
- `eficiencia = speedup / N`. No siempre se acerca a 1, porque hay sobrecostos, competencia por CPU/RAM y disco.
- Los **bytes y hashes finales deben coincidir** entre configuraciones; el orden de finalización puede diferir.
- Se mide **transformación + carga Parquet con fuentes ya descargadas**; no incluye la descarga inicial. El rendimiento depende de tu equipo y del tamaño de imágenes/audios.

**HAZ ESTO AHORA:** ejecuta 8F. Debes ver una tabla con medianas y dos gráficas medidas. El sistema valida la igualdad de los SHA y genera `informe_concurrencia_s11.zip` sin redistribuir archivos multimedia.

**Cómo se lee:** si la mediana con 4 hilos es mayor que con 1, tu carga puede estar saturando CPU o E/S. **Qué nos dice:** cuál configuración rindió mejor en estas ejecuciones. **Qué NO permite concluir:** que esa configuración sea universal ni que se trate de cuatro nodos Spark. **Error común:** calcular el speedup con tiempos de descargas diferentes o atribuir automáticamente la mejora al número de hilos.

**Si estás con tiempo limitado:** completa las tres configuraciones con las **42 fuentes de la ruta base**. El nivel `ensayo` queda para profundización.

**Volumen y memoria del proceso:** la columna `mib_per_second` suma bytes RAW de fuentes terminadas una sola vez más bytes derivados. `peak_rss_cumulative_mb` es el pico residente acumulado del **proceso**, de modo que aumenta con ejecuciones anteriores y NO es un consumo aislado atribuible a 1, 2 o 4 threads. Para comparar memoria de modo justo se requieren procesos nuevos por configuración.

**Descarga del informe:** después de ejecutar 8F activa `DESCARGAR_REPORTE_THREADS` para guardar los CSV y el manifiesto de benchmarks (no incluye originales ni audios). En VS Code el archivo queda dentro de `outputs/etl_live/benchmark/`.

**Código esencial que debes entender:** `ThreadPoolExecutor(max_workers=2)` crea hasta dos trabajadores; `submit(tarea, ...) ` coloca una tarea; `future.result()` recoge el resultado o propaga su error. Verás el código en la nueva práctica **8E-C**, ejecutada con dos archivos reales antes del benchmark grande.


In [ ]:
#@title 8F · Comparar 1, 2 y 4 threads · mediana y eficiencia
import os
REPETICIONES_THREADS=3 #@param [1,2,3] {type:"raw"}
if os.environ.get("S11_CI_FAST")=="1":
    REPETICIONES_THREADS=1 # solo ejecución de QA sin interfaz
print("BENCHMARK REAL: mismos RAW, perfiles y métricas, variando únicamente threads")
BENCHMARK_S11=motor_live.run_benchmark(
    RAIZ,IMAGENES_LIVE,AUDIOS_LIVE,workers=(1,2,4),
    repeats=REPETICIONES_THREADS,batch_size=LOTE_LIVE,core=core)
TABLA_THREADS=BENCHMARK_S11["summary"]
assert len(TABLA_THREADS)==3
assert BENCHMARK_S11["measurements"]["fingerprint"].nunique()==1
print("RESULTADOS REALES: mediana del tiempo y speedup (sin suponer que habrá mejora)")
display(TABLA_THREADS)
display(BENCHMARK_S11["measurements"][
    ["rep","workers","elapsed_seconds","derivatives_per_second",
     "mib_per_second","peak_rss_cumulative_mb"]])
fig,axes=plt.subplots(1,2,figsize=(11,3))
axes[0].bar(TABLA_THREADS["workers"].astype(str),TABLA_THREADS["seconds_median"])
axes[0].set_xlabel("Threads");axes[0].set_ylabel("Segundos (mediana)")
axes[0].set_title("Tiempo total medido")
axes[1].bar(TABLA_THREADS["workers"].astype(str),TABLA_THREADS["speedup"])
axes[1].set_xlabel("Threads");axes[1].set_ylabel("Speedup observado")
axes[1].set_title("Aceleración (puede ser inferior a 1)")
plt.tight_layout();plt.show()
REPORTE_THREADS=motor_live.export_benchmark(BENCHMARK_S11)
print("INFORME CONCURRENCIA LISTO:",REPORTE_THREADS)
DESCARGAR_REPORTE_THREADS=False #@param {type:"boolean"}
if DESCARGAR_REPORTE_THREADS:
    if EN_COLAB:
        from google.colab import files as archivos_colab
        archivos_colab.download(str(REPORTE_THREADS))
    else:
        print("Abre en VS Code esta ruta:",REPORTE_THREADS)


## 8G · La prueba que distingue un sistema fiable: fallos y cancelación

**Experimento seguro:** el motor provoca un error **antes de transformar** un archivo elegido. No modifica los RAW; registra un `retry` y vuelve a intentar. Luego dispara una cancelación al recibir el primer evento `processed`, por lo que pueden acabar las tareas que ya estaban en vuelo, pero **no** se genera un expediente marcado como completo.

**HAZ ESTO AHORA:** ejecuta la celda 8G. No dependas de que el botón responda durante un cálculo CPU que no suelta el GIL: esta cancelación programática está probada y utiliza el mismo indicador `cancel_event` que el tablero.

**Salida esperada:** `REINTENTO VERIFICADO`, `CANCELACIÓN SEGURA` y `RECUPERACIÓN VERIFICADA`.

**Cómo se lee:** el manifiesto `cancelled` conserva el estado pero NO tiene huella global de éxito. **Qué nos dice:** el motor evita aceptar una ejecución incompleta. **Qué NO permite concluir:** que puede detener instantáneamente una operación de biblioteca nativa. **Error común:** interpretar un archivo derivado que quedó en disco como una carga completa.


In [ ]:
#@title 8G · Error + reintento + cancelación + recuperación sobre una copia lógica
from threading import Event
IMAGES_TEST=IMAGENES_RAW[:4]
AUDIOS_TEST=AUDIOS_RAW[:3]
REINTENTO=motor_live.run_live(RAIZ,IMAGES_TEST,AUDIOS_TEST,workers=2,batch_size=8,
                              label="prueba_reintento",inject_once_at=1,core=core)
assert REINTENTO["status"]=="complete" and REINTENTO["manifest"]["retries"]==1
assert REINTENTO["verified"]["ok"]
print("REINTENTO VERIFICADO: un fallo didáctico, una recuperación, cero RAW alterados")
senal_cancel=Event()
def solicitar_cancelacion(evento):
    if evento["state"]=="processed":senal_cancel.set()
CANCELADO=motor_live.run_live(RAIZ,IMAGES_TEST,AUDIOS_TEST,workers=2,batch_size=8,
                             label="prueba_cancelacion",cancel_event=senal_cancel,
                             on_event=solicitar_cancelacion,core=core)
assert CANCELADO["status"]=="cancelled" and CANCELADO["manifest"]["fingerprint"] is None
assert not motor_live.verify_run(RAIZ,CANCELADO["manifest_path"],core=core)["ok"]
print("CANCELACIÓN SEGURA: los derivados parciales NO son un expediente completo")
RECUPERADO=motor_live.run_live(RAIZ,IMAGES_TEST,AUDIOS_TEST,workers=2,batch_size=8,
                              label="prueba_recuperacion",core=core)
assert RECUPERADO["verified"]["ok"]
assert RECUPERADO["manifest"]["fingerprint"]==REINTENTO["manifest"]["fingerprint"]
print("RECUPERACIÓN VERIFICADA: mismos originales y mismo resultado")


## 8H · Puente con la herramienta web E2E (opcional)

Si ejecutaste el ETL de hasta seis archivos en la presentación, descarga su ZIP verificable y **si quieres** impórtalo en el cuaderno, sin volver a digitalizar los originales. La herramienta web exporta el contrato ya existente `s10-asset-v2`, que se convierte mediante el puente oficial probado; **no** se mezcla silenciosamente con las 42 entradas de los corpus reales S11.

**HAZ ESTO AHORA:** activa `IMPORTAR_ZIP_WEB`, descarga el ZIP de la herramienta S11 y selecciónalo en el diálogo de Colab. En VS Code copia el archivo a `work_s11.../etl_usuario.zip` según la ruta indicada. El validador rechaza relaciones falsas con artículos, rutas inseguras y hashes inconsistentes.

**Salida esperada:** `PAQUETE WEB VERIFICADO` y una tabla de metadatos. **Cómo se lee:** el `asset_id` reconoce el mismo RAW; el `profile` distingue derivados. **Qué nos dice:** la carga se puede consultar en ambos entornos. **Qué NO permite concluir:** que una fotografía subida por el estudiante corresponde a un contrato SECOP. **Error común:** inventar `article_id` para unir dos fuentes no relacionadas.


In [ ]:
#@title 8H · Importar el ZIP del E2E web al mismo cuaderno (opcional)
IMPORTAR_ZIP_WEB=False #@param {type:"boolean"}
if IMPORTAR_ZIP_WEB:
    puente_web=cargar_modulo_s11("s10_asset_bridge")
    if EN_COLAB:
        from google.colab import files as archivos_colab
        subidos=archivos_colab.upload()
        if len(subidos)!=1:raise ValueError("Selecciona un ZIP únicamente")
        nombre,contenido=next(iter(subidos.items()))
        if not nombre.lower().endswith(".zip"):raise ValueError("Se espera ZIP")
        ruta_zip=RAIZ/"etl_usuario.zip"
        ruta_zip.write_bytes(contenido)
    else:
        ruta_zip=RAIZ/"etl_usuario.zip"
        if not ruta_zip.is_file():
            raise FileNotFoundError(f"Copia tu ZIP del navegador aquí: {ruta_zip}")
    importe_web=puente_web.importar_zip_etl_web(ruta_zip,RAIZ)
    filas_web=pd.read_json(importe_web["catalogo"],lines=True)
    assert importe_web["cantidad"]==len(filas_web)
    assert filas_web["source_kind"].isin(["uploaded_by_student","generated_for_course"]).all()
    print(importe_web["verificacion"],importe_web["cantidad"],"activos; sin relaciones editoriales ficticias")
    display(filas_web[["asset_id","modality","source_kind","profile","metric_name","metric_value"]])
else:
    print("El expediente oficial S11 sigue íntegro; puedes probar E2E en la presentación.")


## 9 · Entrega verificable y autocomprobación

**HAZ ESTO AHORA.** Genera el ZIP con `raw/` (incluye 32 originales y 10 fuentes WAV), `derived/` (42 transformaciones), catálogo JSONL/Parquet, resumen, decisiones y manifiesto. No incluye los ~39 MB del Parquet descargado como caché: incluye diez WAV fuente originales comprobados.

**Salida:** `ENTREGA · ZIP VERIFICADO:` y `AUTOCOMPROBACIÓN: LISTO (9/9)` si cambiaste las decisiones 0 = pendiente. Un estado INCOMPLETO te dice qué falta y no se inventa una calificación.

**Cómo se lee:** los nueve controles miran archivos y variables de esta ejecución. **Qué nos dice:** que existe un expediente técnico repetible. **Qué NO permite concluir:** dominio conceptual sin sustentación ni que sea una implementación distribuida. **Error común:** entregar una captura de pantalla en lugar del paquete verificable.


In [ ]:
#@title 9A · Generar un paquete ZIP real sin subir datos privados
ARCHIVO_FINAL=core.empaquetar(RAIZ)
print("SHA-256 ZIP:",core.sha256(ARCHIVO_FINAL))
DESCARGAR_ZIP=False #@param {type:"boolean"}
if DESCARGAR_ZIP:
    if EN_COLAB:
        from google.colab import files as archivos_colab
        archivos_colab.download(str(ARCHIVO_FINAL))
    else:
        print("En VS Code encontrarás el ZIP en:",ARCHIVO_FINAL)
else:
    print("ZIP disponible en el panel de archivos:",ARCHIVO_FINAL)


In [ ]:
#@title 9B · Verificar el expediente y tus decisiones (sin respuestas abiertas)
VERIFICACION_FINAL=core.verificar(RAIZ)
CONTROLES={
 "01 · 32 RAW imágenes":len(IMAGENES_RAW)==32,
 "02 · 10 RAW voces reales":len(AUDIOS_RAW)==10,
 "03 · 32 transformaciones imagen":len(IMAGENES_T)==32,
 "04 · 10 transformaciones audio":len(AUDIOS_T)==10,
 "05 · Originales verificados":all(core.sha256(RAIZ/x["original_path"])==x["original_sha256"] for x in [*IMAGENES_RAW,*AUDIOS_RAW]),
 "06 · Catálogo 42 filas":CATALOGO["resumen"]["total"]==42,
 "07 · Manifiesto íntegro":VERIFICACION_FINAL["ok"],
 "08 · Corrupción detectada":PRUEBA_FALLO=="CORRUPCION_DETECTADA",
 "09 · Alternativa y límite elegidos":ALTERNATIVA_DESCARTADA in (1,2,3) and LIMITE_ELEGIDO in (1,2,3),
}
for etiqueta,ok in CONTROLES.items():print(("✓" if ok else "PENDIENTE"),etiqueta)
print(f"AUTOCOMPROBACIÓN: {'LISTO' if all(CONTROLES.values()) else 'INCOMPLETO'} ({sum(CONTROLES.values())}/9)")


## Para llevar · herramienta de consulta

| Herramienta | Entrada | Salida | Error frecuente |
|---|---|---|---|
| `extraer_imagenes` | Snapshot de fotos 32/24 | RAW y SHA | Descargar de nuevo del medio |
| `extraer_audios` | MINDS-14 es-ES CC BY 4.0 | 10 WAV fuente | Suponer que son audios de noticias |
| `transformar_imagenes` | WEBP + perfil + lado | PNG, luminancia, histograma | Confundir brillo con contenido |
| `transformar_audios` | WAV fuente + ganancia | PCM16 16 kHz, RMS, ZCR, espectro | Confundir RMS con transcripción |
| `cargar_catalogo` | 42 derivados, decisiones | JSONL, Parquet | Inventar claves de unión |
| `verificar` | Manifiesto y archivos | Resultado comprobable | Tomar hash como autenticidad |
| `empaquetar` | Expediente aprobado | ZIP portable | Olvidar conservarlo |

**Conclusión que puede defender Laura:** preparó y verificó 42 activos reales de dos modalidades, preservó los originales y separó sus orígenes; todavía no demostró procesamiento distribuido, reconocimiento semántico ni irregularidades de contratación. Los resultados ya están listos para investigar particionado y escala con Dask/Spark.

**Reconocimiento de fuentes:** fotos previamente incorporadas al snapshot del curso con sus créditos y restricciones editoriales; voces PolyAI/MINDS-14 es-ES, licencia CC BY 4.0. No se realiza minería automatizada nueva del sitio del periódico.

**PDA:** el docente decidió estudiar **audio e imágenes** en S11 y no video. Queda constancia de que la secuencia oficial de S11 originalmente menciona audio/video. Esta práctica es formativa y no sustituye el Taller 2 del PDA.
**Escala experimental (8B–8C):** misma tubería con 102, 332 o 1.432 RAW distintos y 174, 564 o 2.464 derivados. Elige lote de 8 a 128 fuentes, reabre particiones y mide throughput. Es procesamiento **local**, no un benchmark de clúster. Esta ampliación mantiene el expediente obligatorio de 42 filas intacto.

**Instrumentación y comparación S11:** 8D carga el tablero, 8E muestra eventos verdaderos por archivo, 8F mide speedup y eficiencia mediante medianas, 8G prueba recuperación, 8H importa opcionalmente el ZIP E2E de la presentación. Los únicos dos recursos del LMS siguen siendo presentación y cuaderno.


### Diccionario final del expediente (retoma las prácticas 0C, 6B y 6C)

| Ruta o extensión | Función en el encargo | Cómo verificarla |
|---|---|---|
| `raw/` | Prueba de lo que se recibió | SHA igual al registrado |
| `derived/` | Resultado técnico transformado | SHA del derivado y perfil |
| `cache/` | Fuente intermedia verificada reutilizable | SHA del Parquet original |
| `outputs/catalogo.jsonl` | Un registro JSON por línea; fácil de anexar e inspeccionar | `json.loads` en cada línea |
| `outputs/catalogo.parquet` | Tabla columnar tipada para consultas | `pd.read_parquet` |
| `outputs/decision.json` | Opciones de proceso seleccionadas | `json.load` |
| `manifest_s11.json` | Lista de archivos, conteos y hashes esperados | Validación `core.verificar` |
| `outputs/etl_live/events.jsonl` | Eventos reales de ejecución por tarea | Filtrar `state` y `seconds` |
| `expediente_*.zip` | Contenedor portable del trabajo permitido | Reabrir/verificar antes de compartir |

**La prueba de aprendizaje:** puedes señalar un archivo, localizar su fila, explicar por qué fue transformado, abrir original/derivado y verificar integridad. El rendimiento y los threads se analizan **después** de construir esa capacidad.
